### Analytical Objective
For each key numeric column (price, freight_value, payment_value, delivery time, delay): what's its shape, center, spread, and outlier profile — and what does that imply for cleaning?

In [ ]:
import os
from google.cloud import bigquery

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"E:\reprompt-olist\olist-analytics-500708-500708-0ee88429fe90.json"

client = bigquery.Client()
print("BigQuery client created successfully")

BigQuery client created successfully


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import db_dtypes

In [ ]:
PROJECT_ID = "olist-analytics-500708-500708" 
client = bigquery.Client(project=PROJECT_ID)

In [9]:
# ── 1. Pull a small, order-grain aggregate from BigQuery (SQL does the heavy lifting) ──
query = """
SELECT
  o.order_id,
  DATE_DIFF(DATE(SAFE_CAST(o.order_delivered_customer_date AS TIMESTAMP)),
            DATE(SAFE_CAST(o.order_purchase_timestamp AS TIMESTAMP)), DAY)   AS delivery_days,
  DATE_DIFF(DATE(SAFE_CAST(o.order_delivered_customer_date AS TIMESTAMP)),
            DATE(SAFE_CAST(o.order_estimated_delivery_date AS TIMESTAMP)), DAY) AS delay_days,
  oi.order_value,
  oi.freight_total
FROM `olist_raw.orders` o
JOIN (
  SELECT order_id, SUM(price) AS order_value, SUM(freight_value) AS freight_total
  FROM `olist_raw.order_items`
  GROUP BY order_id
) oi USING (order_id)
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
"""
df = client.query(query).to_dataframe()
print(df.shape)
# df.describe()
df.head(10)

e:\reprompt-olist\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(96470, 5)


,order_id,delivery_days,delay_days,order_value,freight_total
0,bfbd0f9bdef84302105ad712db648a6c,55,36,134.97,8.49
1,98974b076b01553d49ee6467905675a7,44,-7,109.90,11.23
2,c4b41c36dd589e901f6879f25a74ec1d,36,-15,9.90,8.72
3,d2292ff2201e74c5db154d1b7ae68cbb,30,-21,49.90,11.74
4,95e01270fcbae9863423400103359279,31,-20,86.99,28.23
5,ed8c7b1b3eb256c70ce0c74231e1da88,45,-6,89.90,24.87
6,5cc475c7c03290048eb2e742cd64cb5e,69,18,71.00,11.69
7,6b3ee7697a02619a0ace2b3f0aa46bde,48,-3,57.00,8.77
8,3b2ca3293a7ce539ea2379d704fa37ce,43,-8,169.90,14.97
9,b2f92b2f7047cd8b35580d629d7b3bfb,43,-8,59.90,11.31


In [10]:
df.isna().sum()

order_id         0
delivery_days    0
delay_days       0
order_value      0
freight_total    0
dtype: int64

In [5]:
# ── 2. Central tendency, spread, shape per column ──
NUMERIC_COLS = ["order_value", "freight_total", "delivery_days", "delay_days"]

def profile_column(s: pd.Series) -> dict:
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return {
        "mean": s.mean(), "median": s.median(),
        "sd": s.std(), "iqr": iqr,
        "skew": stats.skew(s), "kurtosis": stats.kurtosis(s),
        "lower_fence": lower_fence, "upper_fence": upper_fence,
        "pct_outside_fence": round(((s < lower_fence) | (s > upper_fence)).mean() * 100, 2),
    }

findings = {col: profile_column(df[col]) for col in NUMERIC_COLS}
findings_df = pd.DataFrame(findings).T
print(findings_df)
findings_df.to_csv("../reports/05_distribution_findings_raw.csv")


                     mean  median          sd     iqr       skew    kurtosis  \
order_value    137.040001   86.50  209.052608  104.00   9.885246  276.981548   
freight_total   22.785798   17.17   21.559959   10.17  12.276711  585.947726   
delivery_days   12.496849   10.00    9.555071    9.00   3.826058   39.229376   
delay_days     -11.875889  -12.00   10.182105   10.00   2.016132   28.068805   

               lower_fence  upper_fence  pct_outside_fence  
order_value       -110.100      305.900               7.94  
freight_total       -1.405       39.275              10.05  
delivery_days       -6.500       29.500               4.90  
delay_days         -32.000        8.000               4.45  


In [12]:
# ── 3. Visualize: histogram + boxplot + Q-Q per column ──
for col in NUMERIC_COLS:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    df[col].dropna().hist(bins=50, ax=axes[0]); axes[0].set_title(f"{col} — histogram")
    axes[1].boxplot(df[col].dropna()); axes[1].set_title(f"{col} — boxplot")
    stats.probplot(df[col].dropna(), dist="norm", plot=axes[2]); axes[2].set_title(f"{col} — Q-Q plot")
    plt.tight_layout()
    plt.savefig(f"../reports/figures/05_{col}.png")
    plt.close()

In [13]:
# ── 4. Normality check (Q-Q is primary; formal test is secondary at this n — Appendix B.12) ──
for col in NUMERIC_COLS:
    sample = df[col].dropna().sample(min(5000, len(df)), random_state=42)  # Shapiro caps ~5000
    stat, p = stats.shapiro(sample)
    print(col, "Shapiro p =", p, "-> not formally normal" if p < 0.05 else "-> approx normal")

order_value Shapiro p = 4.680011596740469e-83 -> not formally normal
freight_total Shapiro p = 1.6217793981128654e-81 -> not formally normal
delivery_days Shapiro p = 1.1358627603113063e-64 -> not formally normal
delay_days Shapiro p = 2.2491303238090486e-50 -> not formally normal


In [15]:
# ── 5. Write the implication for each column (fill manually after reviewing plots) ──
IMPLICATIONS = {
    "order_value":   "Extreme right skew, most orders below R$500 but tail to R$14,000, "
                    "very similar shape to freight_total.",
    "freight_total":  "Extreme right skew, almost all orders below R$100 but tail extends to R$1,800, "
                    "Q-Q shows massive right-tail deviation.",
    "delivery_days":  "Strong right skew, majority cluster 1-50 days, long tail to 220 days, "
                        "Q-Q confirms severe non-normality.",
    "delay_days":     "near-normal centered around -12 (most orders arrive early), "
                      "fat right tail up to +190 days, heavy tails both sides confirmed by Q-Q.",
}
for col, note in IMPLICATIONS.items():
    print(f"\n{col}:\n  {note}")

print("\nDone. Paste findings_df + implications into reports/05_distribution_findings.md")


order_value:
  Extreme right skew, most orders below R$500 but tail to R$14,000, very similar shape to freight_total.

freight_total:
  Extreme right skew, almost all orders below R$100 but tail extends to R$1,800, Q-Q shows massive right-tail deviation.

delivery_days:
  Strong right skew, majority cluster 1-50 days, long tail to 220 days, Q-Q confirms severe non-normality.

delay_days:
  near-normal centered around -12 (most orders arrive early), fat right tail up to +190 days, heavy tails both sides confirmed by Q-Q.

Done. Paste findings_df + implications into reports/05_distribution_findings.md
